# Task 2 — Loan Approval Prediction

Place the official loan dataset in `data/raw/` with `loan` in its filename. Confirm the target column from the assignment before training.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
DATA = ROOT / "data" / "raw"
files = list(DATA.glob("*.csv")) + list(DATA.glob("*.xlsx")) + list(DATA.glob("*.json"))
print("Data files found:", [f.name for f in files])
print("Put the correct task dataset in data/raw/ before interpreting results.")

def load_data(keywords):
    matches = [f for f in files if any(k in f.name.lower() for k in keywords)]
    if not matches:
        print("DATASET MISSING for:", keywords)
        return None
    f = matches[0]
    print("Loading:", f.name)
    if f.suffix.lower() == ".csv":
        return pd.read_csv(f)
    if f.suffix.lower() == ".xlsx":
        return pd.read_excel(f)
    return pd.read_json(f)

def overview(df):
    if df is None:
        return
    print("Shape:", df.shape)
    display(df.head())
    display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("missing"))
    display(df.describe(include="all").T)

df = load_data(['loan'])
overview(df)

Data files found: []
Put the correct task dataset in data/raw/ before interpreting results.
DATASET MISSING for: ['loan']


## Baseline classification model

In [2]:
if df is not None:
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

    candidates = [c for c in df.columns if any(
        x in c.lower() for x in ["loan_status", "approved", "approval", "target"]
    )]
    print("Possible target columns:", candidates)
    if not candidates:
        print("TARGET NOT IDENTIFIED. Set target_col to the exact target column from the assignment.")
    else:
        target_col = candidates[0]
        work = df.dropna(subset=[target_col]).copy()
        X, y = work.drop(columns=[target_col]), work[target_col]
        if y.nunique() < 2:
            print("Target must contain at least two classes.")
        else:
            numeric_cols = X.select_dtypes(include="number").columns.tolist()
            cat_cols = X.select_dtypes(exclude="number").columns.tolist()
            pre = ColumnTransformer([
                ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                                  ("scale", StandardScaler())]), numeric_cols),
                ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                                  ("encode", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
            ])
            model = Pipeline([("preprocess", pre),
                              ("classifier", RandomForestClassifier(
                                  n_estimators=200, random_state=42,
                                  class_weight="balanced"))])
            strat = y if y.value_counts().min() >= 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=strat)
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            print("Target:", target_col)
            print(classification_report(y_test, pred, zero_division=0))
            print("Confusion matrix:\n", confusion_matrix(y_test, pred))
            print("Suggested threshold must be selected using validation data, costs of false approvals/rejections, and calibration—not guessed.")
else:
    print("Dataset missing. No model metrics or threshold can be truthfully reported yet.")

Dataset missing. No model metrics or threshold can be truthfully reported yet.


## Threshold selection and limitations

Select the operating threshold using a validation set, business costs, fairness checks and calibration. Do not use the test set repeatedly to tune it. This model is a learning exercise, not a real lending decision system.